# Japanese Font via `.fonts`, fontconfig, and `matplotlibrc`

This notebook downloads Noto Sans JP into the current directory's `.fonts` directory, writes a local fontconfig file, and checks whether Matplotlib can render Japanese text through `matplotlibrc`.

Run the setup cell first. If Matplotlib was already imported in this kernel, restart the kernel after the setup cell and then continue from the environment cell.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap
import urllib.request

cwd = Path.cwd()
font_dir = cwd / ".fonts"
fontconfig_file = cwd / ".fontconfig" / "fonts.conf"
matplotlibrc = cwd / "matplotlibrc"

font_url = "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/NotoSansJP%5Bwght%5D.ttf"
license_url = "https://raw.githubusercontent.com/google/fonts/main/ofl/notosansjp/OFL.txt"
font_path = font_dir / "NotoSansJP[wght].ttf"
license_path = font_dir / "OFL.txt"

font_dir.mkdir(parents=True, exist_ok=True)
fontconfig_file.parent.mkdir(parents=True, exist_ok=True)

for url, path in [(font_url, font_path), (license_url, license_path)]:
    if not path.exists() or path.stat().st_size == 0:
        with urllib.request.urlopen(url) as response:
            path.write_bytes(response.read())
    print(path, path.stat().st_size)

fontconfig_file.write_text(textwrap.dedent(f"""\
<?xml version="1.0"?>
<!DOCTYPE fontconfig SYSTEM "urn:fontconfig:fonts.dtd">
<fontconfig>
  <dir>{font_dir}</dir>
</fontconfig>
"""), encoding="utf-8")

matplotlibrc.write_text(textwrap.dedent("""\
font.family: sans-serif
font.sans-serif: Noto Sans JP, DejaVu Sans
axes.unicode_minus: False
"""), encoding="utf-8")

fc_cache = shutil.which("fc-cache")
if fc_cache:
    subprocess.run([fc_cache, "-f", "-v", str(font_dir)], check=False)
else:
    print("fc-cache was not found")

print("fontconfig file:", fontconfig_file)
print("matplotlibrc:", matplotlibrc)

In [ ]:
import os
from pathlib import Path

cwd = Path.cwd()
fontconfig_file = cwd / ".fontconfig" / "fonts.conf"

os.environ["FONTCONFIG_FILE"] = str(fontconfig_file)

print("cwd:", cwd)
print("FONTCONFIG_FILE:", os.environ["FONTCONFIG_FILE"])
print("matplotlib already imported:", "matplotlib" in __import__("sys").modules)
print("matplotlibrc exists:", (cwd / "matplotlibrc").exists())
print((cwd / "matplotlibrc").read_text(encoding="utf-8"))

In [ ]:
import os
import shutil
import subprocess

fc_match = shutil.which("fc-match")
if fc_match:
    result = subprocess.run(
        [fc_match, "Noto Sans JP"],
        env=os.environ.copy(),
        text=True,
        capture_output=True,
        check=False,
    )
    print(result.stdout)
    print(result.stderr)
else:
    print("fc-match was not found")

In [ ]:
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

print("Matplotlib:", matplotlib.__version__)
print("Backend:", matplotlib.get_backend())
print("matplotlibrc loaded from:", matplotlib.matplotlib_fname())
print("font.family:", plt.rcParams["font.family"])
print("font.sans-serif:", plt.rcParams["font.sans-serif"][:5])
print("cache dir:", matplotlib.get_cachedir())

for font in fm.fontManager.ttflist:
    if "Noto Sans JP" in font.name:
        print("found:", font.name, font.fname)

try:
    print("findfont:", fm.findfont("Noto Sans JP", fallback_to_default=False))
except Exception as exc:
    print("findfont failed:", type(exc).__name__, exc)

In [ ]:
from pathlib import Path

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

font_path = Path.cwd() / ".fonts" / "NotoSansJP[wght].ttf"
font_name = fm.FontProperties(fname=font_path).get_name()

fm.fontManager.addfont(font_path)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = [font_name, "DejaVu Sans"]

print("font file:", font_path)
print("internal font name:", font_name)
print("findfont:", fm.findfont(font_name, fallback_to_default=False))

In [ ]:
x = [1, 2, 3, 4]
y = [10, 18, 13, 22]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, y, marker="o", linewidth=2)
ax.set_title("日本語フォントの検証")
ax.set_xlabel("回数")
ax.set_ylabel("値")
ax.grid(True)
plt.show()